# 05 - Búsqueda Híbrida, Generación RAG y Evaluación

Este notebook implementa y compara tres configuraciones de recuperación:

- **Configuración A:** Búsqueda lexical con BM25
- **Configuración B:** Búsqueda vectorial semántica (Qdrant)
- **Configuración C:** Búsqueda híbrida (BM25 + Vectorial + RRF)

Finalmente conecta Gemini API para generación de respuestas y evalúa cada configuración con métricas de recuperación y calidad.

**Requisito:** Haber ejecutado correctamente los notebooks 01 al 04.

## 0. Instalación de dependencias

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q --upgrade pip
!pip uninstall -y transformers FlagEmbedding torchvision torchaudio torchcodec
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2
!pip install -q FlagEmbedding==1.2.11
!pip install -q rank-bm25 google-generativeai qdrant-client pandas numpy tqdm scikit-learn
print("✅ Instalación completa — reinicia la sesión antes de continuar")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:00
Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Found existing installation: torchcodec 0.11.0+cpu
Uninstalling torchcodec-0.11.0+cpu:
  Successfully uninstalled torchcodec-0.11.0+cpu
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires transformers, which is not installed.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
  Installing build dependencies ... done
  Getting requiremen

## 1. Importación de librerías

In [3]:
import json
import time
import gc
import os
import re
import math
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import google.generativeai as genai

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 2. Configuración de rutas y API Key

La API Key de Gemini se carga desde los **Secrets de Colab** de forma segura.
Nunca escribas la clave directamente en el código.

In [4]:
from pathlib import Path

# Ruta fija a tu proyecto en Drive
ROOT_DIR    = Path("/content/drive/MyDrive/RAG_Normativa_Ambiental")
DATA_DIR    = ROOT_DIR / "data"
CHUNKS_DIR  = DATA_DIR / "chunks"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_JSONL_PATH  = CHUNKS_DIR / "chunks_normativa_v1.jsonl"
CORPUS_CSV_PATH    = METADATA_DIR / "corpus_normativo_ambiental.csv"
PREGUNTAS_CSV_PATH = METADATA_DIR / "banco_preguntas_evaluacion.csv"

# Verificar que existen
print(f"Chunks:    {CHUNKS_JSONL_PATH.exists()}")
print(f"Corpus:    {CORPUS_CSV_PATH.exists()}")
print(f"Preguntas: {PREGUNTAS_CSV_PATH.exists()}")

Chunks:    True
Corpus:    True
Preguntas: True


In [5]:
# ── API Key de Gemini desde Secrets de Colab ──
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    print("API Key cargada correctamente desde Secrets.")
except Exception:
    import os
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    print("API Key cargada desde variable de entorno.")

if not GEMINI_API_KEY:
    raise ValueError("No se encontró GEMINI_API_KEY. Agrégala en Secrets de Colab.")

genai.configure(api_key=GEMINI_API_KEY)
print("Gemini configurado correctamente.")

API Key cargada correctamente desde Secrets.
Gemini configurado correctamente.


## 3. Carga de chunks

In [6]:
chunks = []
with open(CHUNKS_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

chunks_df = pd.DataFrame(chunks)
print(f"Total chunks cargados: {len(chunks_df)}")
print(f"Columnas: {list(chunks_df.columns)}")
chunks_df.head(3)

Total chunks cargados: 1057
Columnas: ['id_chunk', 'id_documento', 'archivo_pdf', 'archivo_txt', 'titulo_documento', 'tipo_norma', 'numero_norma', 'entidad_emisora', 'fecha_publicacion', 'tema_principal', 'subtema', 'fuente_oficial', 'url_fuente', 'estado_vigencia', 'prioridad', 'tipo_chunk', 'seccion', 'titulo_seccion', 'capitulo', 'anexo', 'pagina_inicio', 'pagina_fin', 'orden_chunk', 'suborden', 'n_palabras', 'n_caracteres', 'texto']


,id_chunk,id_documento,archivo_pdf,archivo_txt,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,...,titulo_seccion,capitulo,anexo,pagina_inicio,pagina_fin,orden_chunk,suborden,n_palabras,n_caracteres,texto
0,DOC-001_CHK-0001,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,1,453,2860,"NORMAS LEGALES El Peruano Lima, jueves 6 de se..."
1,DOC-001_CHK-0001_P02,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,2,295,1878,así como en el proceso al que se reﬁ ere el De...
2,DOC-001_CHK-0002,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,2,1,58,402,Artículo 1º.- Del objeto de la norma\nApruébes...


## 4. Carga del modelo de embeddings y conexión a Qdrant

In [7]:
from FlagEmbedding import BGEM3FlagModel

print("Cargando modelo BGE-M3...")
embedding_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("Modelo cargado correctamente.")

Cargando modelo BGE-M3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Modelo cargado correctamente.


/usr/local/lib/python3.12/dist-packages/FlagEmbedding/BGE_M3/modeling.py:335: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  colbert_state_dict = torch.load(os.path.join(mode

In [8]:
import os
from pathlib import Path

QDRANT_STORAGE = Path("/content/drive/MyDrive/RAG_Normativa_Ambiental/qdrant_storage")

# Eliminar el archivo de lock
lock_file = QDRANT_STORAGE / ".lock"
if lock_file.exists():
    os.remove(lock_file)
    print("Lock eliminado.")
else:
    print("No se encontró lock, buscando...")
    # A veces tiene otro nombre
    for f in QDRANT_STORAGE.iterdir():
        print(f)

Lock eliminado.


In [9]:
from qdrant_client import QdrantClient
from pathlib import Path

QDRANT_STORAGE = Path("/content/drive/MyDrive/RAG_Normativa_Ambiental/qdrant_storage")

client = QdrantClient(path=str(QDRANT_STORAGE))
collections = [c.name for c in client.get_collections().collections]
print(f"Qdrant con persistencia. Colecciones: {collections}")

COLLECTION_NAME = "normativa_ambiental_chunks_v1"  # ← solo esto cambia
TOP_K = 5

Qdrant con persistencia. Colecciones: ['normativa_ambiental_chunks_v1']


In [10]:
# ── Función para generar embeddings ──
def encode_texts(texts: list, is_query: bool = False) -> np.ndarray:
    if is_query:
        output = embedding_model.encode(
            texts,
            batch_size=1,
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False
        )
    else:
        output = embedding_model.encode(
            texts,
            batch_size=4,
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False
        )
    vectors = np.array(output["dense_vecs"], dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms

## 5. Configuración A — Búsqueda lexical con BM25

BM25 trabaja con coincidencias exactas de términos. Es útil para consultas con
números de norma, artículos específicos o términos legales exactos.

In [11]:
# ── Tokenización para BM25 ──
def tokenize(text: str) -> list:
    """Tokenización simple: minúsculas, solo palabras alfanuméricas."""
    text = text.lower()
    tokens = re.findall(r'[a-záéíóúüñ0-9]+', text)
    return tokens

# Construir corpus tokenizado
corpus_textos = chunks_df["texto"].fillna("").tolist()
corpus_tokenizado = [tokenize(t) for t in corpus_textos]

# Construir índice BM25
bm25_index = BM25Okapi(corpus_tokenizado)
print(f"Índice BM25 construido con {len(corpus_tokenizado)} documentos.")

Índice BM25 construido con 1057 documentos.


In [12]:
# ── Función de búsqueda BM25 ──
def search_bm25(query: str, top_k: int = TOP_K) -> list:
    """Búsqueda lexical con BM25. Retorna lista de resultados con score."""
    query_tokens = tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            row = chunks_df.iloc[idx]
            results.append({
                "id_chunk":        row.get("id_chunk", ""),
                "id_documento":    row.get("id_documento", ""),
                "titulo_documento":row.get("titulo_documento", ""),
                "texto":           row.get("texto", "")[:300],
                "score_bm25":      float(scores[idx]),
                "score_vectorial": 0.0,
                "score_rrf":       0.0,
                "configuracion":   "A_BM25"
            })
    return results

# ── Prueba rápida ──
print("=== PRUEBA BM25 ===")
test = search_bm25("estándares de calidad ambiental del aire DS 003-2017")
for r in test:
    print(f"  [{r['score_bm25']:.3f}] {r['id_documento']} | {r['titulo_documento'][:60]}")

=== PRUEBA BM25 ===
  [23.973] DOC-008 | Decreto Supremo que establece medidas para operativizar la g
  [21.870] DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) para Aire y e
  [21.464] DOC-002 | Aprueban los Lineamientos para el Fortalecimiento e incorpor
  [21.127] DOC-026 | Establecen el Índice de Calidad del Aire - INCA y crean el S
  [20.585] DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) para Aire y e


## 6. Configuración B — Búsqueda vectorial semántica (Qdrant)

La búsqueda vectorial usa similitud coseno entre el embedding de la consulta
y los embeddings de los chunks indexados en Qdrant.

In [13]:
# ── Función de búsqueda vectorial ──
def search_vectorial(query: str, top_k: int = TOP_K, filtros: dict = None) -> list:
    """Búsqueda semántica usando Qdrant. Soporta filtros por metadatos."""
    query_vector = encode_texts([query], is_query=True)[0]

    from qdrant_client.models import Filter, FieldCondition, MatchValue

    query_filter = None
    if filtros:
        conditions = [
            FieldCondition(key=k, match=MatchValue(value=v))
            for k, v in filtros.items()
        ]
        query_filter = Filter(must=conditions)

    results_raw = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
        query_filter=query_filter
    ).points

    results = []
    for r in results_raw:
        p = r.payload
        results.append({
            "id_chunk":         p.get("id_chunk", ""),
            "id_documento":     p.get("id_documento", ""),
            "titulo_documento": p.get("titulo_documento", ""),
            "texto":            p.get("texto", "")[:300],
            "score_bm25":       0.0,
            "score_vectorial":  float(r.score),
            "score_rrf":        0.0,
            "configuracion":    "B_VECTORIAL"
        })
    return results

# Prueba rápida
print("=== PRUEBA VECTORIAL ===")
test = search_vectorial("estándares de calidad ambiental del aire")
for r in test:
    print(f"  [{r['score_vectorial']:.3f}] {r['id_documento']} | {r['titulo_documento'][:60]}")

=== PRUEBA VECTORIAL ===
  [0.713] DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  [0.688] DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) para Aire y e
  [0.683] DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  [0.680] DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  [0.679] DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) para Aire y e


## 7. Configuración C — Búsqueda híbrida con RRF

**Reciprocal Rank Fusion (RRF)** combina los rankings de BM25 y búsqueda vectorial.
La fórmula es: `RRF(d) = Σ 1 / (k + rank(d))` donde k=60 es el parámetro estándar.

Esto permite aprovechar tanto la precisión léxica de BM25 como la comprensión
semántica de los embeddings.

In [14]:
# ── Función RRF ──
def reciprocal_rank_fusion(rankings: list, k: int = 60) -> dict:
    """
    Fusiona múltiples rankings usando Reciprocal Rank Fusion.
    rankings: lista de listas, cada una con ids ordenados por relevancia.
    Retorna dict {id: score_rrf}
    """
    rrf_scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return rrf_scores


# ── Función de búsqueda híbrida ──
def search_hibrida(query: str, top_k: int = TOP_K,
                   top_k_candidatos: int = 20) -> list:
    """
    Búsqueda híbrida: BM25 + Vectorial fusionados con RRF.
    top_k_candidatos: cuántos candidatos tomar de cada método antes de fusionar.
    """
    # ── BM25: obtener candidatos ──
    query_tokens = tokenize(query)
    bm25_scores  = bm25_index.get_scores(query_tokens)
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:top_k_candidatos]

    bm25_ranking = []
    bm25_score_map = {}
    for idx in bm25_top_idx:
        chunk_id = chunks_df.iloc[idx].get("id_chunk", str(idx))
        bm25_ranking.append(chunk_id)
        bm25_score_map[chunk_id] = float(bm25_scores[idx])

    # ── Vectorial: obtener candidatos ──
    query_vector = encode_texts([query], is_query=True)[0]
    # DESPUÉS - API actual
    vec_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=top_k_candidatos,
        with_payload=True
    ).points

    vec_ranking = []
    vec_score_map = {}
    vec_payload_map = {}
    for r in vec_results:
        chunk_id = r.payload.get("id_chunk", "")
        vec_ranking.append(chunk_id)
        vec_score_map[chunk_id]   = float(r.score)
        vec_payload_map[chunk_id] = r.payload

    # ── RRF: fusionar rankings ──
    rrf_scores = reciprocal_rank_fusion([bm25_ranking, vec_ranking])

    # Ordenar por score RRF
    sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # ── Construir resultados ──
    results = []
    for chunk_id, rrf_score in sorted_chunks:
        # Buscar payload: primero en vectorial, luego en chunks_df
        if chunk_id in vec_payload_map:
            p = vec_payload_map[chunk_id]
            texto = p.get("texto", "")[:300]
            id_doc = p.get("id_documento", "")
            titulo = p.get("titulo_documento", "")
        else:
            row = chunks_df[chunks_df["id_chunk"] == chunk_id]
            if not row.empty:
                r = row.iloc[0]
                texto  = str(r.get("texto", ""))[:300]
                id_doc = str(r.get("id_documento", ""))
                titulo = str(r.get("titulo_documento", ""))
            else:
                texto, id_doc, titulo = "", "", ""

        results.append({
            "id_chunk":         chunk_id,
            "id_documento":     id_doc,
            "titulo_documento": titulo,
            "texto":            texto,
            "score_bm25":       bm25_score_map.get(chunk_id, 0.0),
            "score_vectorial":  vec_score_map.get(chunk_id, 0.0),
            "score_rrf":        rrf_score,
            "configuracion":    "C_HIBRIDA_RRF"
        })
    return results

# ── Prueba rápida ──
print("=== PRUEBA HÍBRIDA RRF ===")
test = search_hibrida("estándares de calidad ambiental del aire DS 003-2017")
for r in test:
    print(f"  [RRF:{r['score_rrf']:.4f} | BM25:{r['score_bm25']:.3f} | VEC:{r['score_vectorial']:.3f}] "
          f"{r['id_documento']} | {r['titulo_documento'][:50]}")

=== PRUEBA HÍBRIDA RRF ===
  [RRF:0.0313 | BM25:21.464 | VEC:0.597] DOC-002 | Aprueban los Lineamientos para el Fortalecimiento 
  [RRF:0.0303 | BM25:20.209 | VEC:0.612] DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  [RRF:0.0302 | BM25:19.925 | VEC:0.617] DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  [RRF:0.0300 | BM25:19.393 | VEC:0.624] DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  [RRF:0.0298 | BM25:21.870 | VEC:0.580] DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) par


## 8. Generación de respuestas con Gemini API

El prompt restringe la respuesta **únicamente** a los fragmentos recuperados.
Si la información no está en el contexto, el sistema lo indica explícitamente.

In [15]:
# ── Modelo Gemini ──
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

PROMPT_TEMPLATE = """Eres un asistente especializado en normativa ambiental peruana.
Responde la pregunta usando ÚNICAMENTE la información de los fragmentos proporcionados.

REGLAS ESTRICTAS:
- Si la respuesta no está en los fragmentos, responde: "La información no se encuentra en los documentos disponibles."
- No inventes artículos, fechas, números de norma ni entidades.
- Cita siempre el documento fuente (id_documento y título).
- Sé claro y preciso.

FRAGMENTOS RECUPERADOS:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:"""


def generar_respuesta(pregunta: str, fragmentos: list) -> dict:
    """Genera respuesta con Gemini basada en los fragmentos recuperados."""
    if not fragmentos:
        return {
            "respuesta": "No se encontraron fragmentos relevantes para responder.",
            "tokens_usados": 0,
            "latencia_seg": 0.0
        }

    # Construir contexto desde fragmentos
    contexto_partes = []
    for i, f in enumerate(fragmentos, 1):
        contexto_partes.append(
            f"[Fragmento {i}]\n"
            f"Documento: {f.get('id_documento','')} - {f.get('titulo_documento','')}\n"
            f"Texto: {f.get('texto','')}\n"
        )
    contexto = "\n".join(contexto_partes)

    prompt = PROMPT_TEMPLATE.format(contexto=contexto, pregunta=pregunta)

    start = time.time()
    try:
        response = gemini_model.generate_content(prompt)
        latencia = time.time() - start
        respuesta_texto = response.text.strip()
        tokens = response.usage_metadata.total_token_count if hasattr(response, 'usage_metadata') else 0
    except Exception as e:
        latencia = time.time() - start
        respuesta_texto = f"Error al generar respuesta: {str(e)}"
        tokens = 0

    return {
        "respuesta":     respuesta_texto,
        "tokens_usados": tokens,
        "latencia_seg":  round(latencia, 3)
    }

print("Función de generación lista.")

Función de generación lista.


In [17]:
# ── Prueba de generación end-to-end ──
pregunta_prueba = "¿Cuáles son los Estándares de Calidad Ambiental para aire según el DS 003-2017-MINAM?"

fragmentos_prueba = search_hibrida(pregunta_prueba)
resultado_prueba  = generar_respuesta(pregunta_prueba, fragmentos_prueba)

print(f"PREGUNTA: {pregunta_prueba}")
print(f"LATENCIA: {resultado_prueba['latencia_seg']} seg")
print(f"\nRESPUESTA:\n{resultado_prueba['respuesta']}")
print(f"\nFUENTES USADAS:")
for f in fragmentos_prueba:
    print(f"  - {f['id_documento']} | {f['titulo_documento'][:60]}")

PREGUNTA: ¿Cuáles son los Estándares de Calidad Ambiental para aire según el DS 003-2017-MINAM?
LATENCIA: 2.485 seg

RESPUESTA:
Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.

FUENTES USADAS:
  - DOC-008 | Decreto Supremo que establece medidas para operativizar la g
  - DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  - DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) para Aire y e
  - DOC-011 | Aprueban Estándares de Calidad Ambiental para Aire
  - DOC-016 | Aprueban Estándares de Calidad Ambiental (ECA) para Aire y e


## 9. Evaluación comparativa de las 3 configuraciones

Se evalúa cada configuración con el banco de preguntas usando:
- **Recall@5:** ¿El documento correcto aparece en los 5 primeros resultados?
- **Precision@5:** ¿Cuántos de los 5 resultados son relevantes?
- **MRR:** ¿En qué posición aparece el primer resultado correcto?
- **Latencia promedio**

In [18]:
# ── Carga del banco de preguntas ──
preguntas_df = pd.read_csv(PREGUNTAS_CSV_PATH)
print(f"Preguntas cargadas: {len(preguntas_df)}")

# Excluir preguntas trampa de las métricas de recuperación
preguntas_eval = preguntas_df[preguntas_df["tipo_pregunta"] != "trampa_alucinacion"].copy()
preguntas_trampa = preguntas_df[preguntas_df["tipo_pregunta"] == "trampa_alucinacion"].copy()

print(f"Preguntas para evaluación de recuperación: {len(preguntas_eval)}")
print(f"Preguntas trampa (alucinación): {len(preguntas_trampa)}")

Preguntas cargadas: 40
Preguntas para evaluación de recuperación: 38
Preguntas trampa (alucinación): 2


In [19]:
# ── Funciones de métricas ──

def get_docs_esperados(doc_esperado_str: str) -> list:
    """Parsea el campo doc_esperado que puede tener múltiples docs separados por coma."""
    if pd.isna(doc_esperado_str) or doc_esperado_str == "NINGUNO":
        return []
    return [d.strip() for d in str(doc_esperado_str).split(",")]


def calcular_recall_at_k(docs_recuperados: list, docs_esperados: list, k: int = 5) -> float:
    """Recall@k: ¿Al menos uno de los docs esperados está en los top-k?"""
    if not docs_esperados:
        return 0.0
    recuperados_ids = set([r["id_documento"] for r in docs_recuperados[:k]])
    hits = len(set(docs_esperados) & recuperados_ids)
    return min(hits / len(docs_esperados), 1.0)


def calcular_precision_at_k(docs_recuperados: list, docs_esperados: list, k: int = 5) -> float:
    """Precision@k: Proporción de resultados relevantes en top-k."""
    if not docs_recuperados or not docs_esperados:
        return 0.0
    recuperados_ids = [r["id_documento"] for r in docs_recuperados[:k]]
    hits = sum(1 for d in recuperados_ids if d in docs_esperados)
    return hits / min(k, len(recuperados_ids))


def calcular_mrr(docs_recuperados: list, docs_esperados: list) -> float:
    """MRR: Inverso de la posición del primer resultado relevante."""
    if not docs_esperados:
        return 0.0
    for rank, r in enumerate(docs_recuperados, start=1):
        if r["id_documento"] in docs_esperados:
            return 1.0 / rank
    return 0.0

print("Funciones de métricas definidas.")

Funciones de métricas definidas.


In [20]:
# ── Evaluación de las 3 configuraciones ──
configuraciones = {
    "A_BM25":       search_bm25,
    "B_VECTORIAL":  search_vectorial,
    "C_HIBRIDA_RRF": search_hibrida
}

resultados_evaluacion = []

for config_nombre, search_fn in configuraciones.items():
    print(f"\nEvaluando configuración: {config_nombre}")
    recall_scores    = []
    precision_scores = []
    mrr_scores       = []
    latencias        = []

    for _, fila in tqdm(preguntas_eval.iterrows(), total=len(preguntas_eval)):
        pregunta      = fila["pregunta"]
        docs_esperados = get_docs_esperados(fila["doc_esperado"])

        start = time.time()
        try:
            fragmentos = search_fn(pregunta, top_k=TOP_K)
        except Exception as e:
            print(f"  Error en {config_nombre} para '{pregunta[:40]}': {e}")
            fragmentos = []
        latencia = time.time() - start

        recall    = calcular_recall_at_k(fragmentos, docs_esperados)
        precision = calcular_precision_at_k(fragmentos, docs_esperados)
        mrr       = calcular_mrr(fragmentos, docs_esperados)

        recall_scores.append(recall)
        precision_scores.append(precision)
        mrr_scores.append(mrr)
        latencias.append(latencia)

        resultados_evaluacion.append({
            "configuracion":  config_nombre,
            "id_pregunta":    fila["id_pregunta"],
            "pregunta":       pregunta,
            "tipo_pregunta":  fila["tipo_pregunta"],
            "dificultad":     fila["dificultad"],
            "doc_esperado":   fila["doc_esperado"],
            "docs_recuperados": ", ".join([r["id_documento"] for r in fragmentos]),
            "recall_at_5":    recall,
            "precision_at_5": precision,
            "mrr":            mrr,
            "latencia_seg":   round(latencia, 3)
        })

    print(f"  Recall@5:    {np.mean(recall_scores):.3f}")
    print(f"  Precision@5: {np.mean(precision_scores):.3f}")
    print(f"  MRR:         {np.mean(mrr_scores):.3f}")
    print(f"  Latencia:    {np.mean(latencias):.3f} seg")

print("\nEvaluación completada.")


Evaluando configuración: A_BM25


  0%|          | 0/38 [00:00<?, ?it/s]

  Recall@5:    0.715
  Precision@5: 0.563
  MRR:         0.651
  Latencia:    0.007 seg

Evaluando configuración: B_VECTORIAL


  0%|          | 0/38 [00:00<?, ?it/s]

  Recall@5:    0.779
  Precision@5: 0.721
  MRR:         0.803
  Latencia:    0.718 seg

Evaluando configuración: C_HIBRIDA_RRF


  0%|          | 0/38 [00:00<?, ?it/s]

  Recall@5:    0.792
  Precision@5: 0.674
  MRR:         0.809
  Latencia:    0.573 seg

Evaluación completada.


## 10. Resumen comparativo de métricas

In [22]:
eval_df = pd.DataFrame(resultados_evaluacion)

# ── Tabla resumen por configuración ──
resumen = eval_df.groupby("configuracion").agg(
    Recall_5    = ("recall_at_5",    "mean"),
    Precision_5 = ("precision_at_5", "mean"),
    MRR         = ("mrr",            "mean"),
    Latencia    = ("latencia_seg",   "mean"),
    N_preguntas = ("id_pregunta",    "count")
).round(4).reset_index()

print("=" * 65)
print("TABLA COMPARATIVA DE CONFIGURACIONES")
print("=" * 65)
print(resumen.to_string(index=False))
print("=" * 65)

# ── Mejor configuración ──
mejor = resumen.loc[resumen["MRR"].idxmax(), "configuracion"]
print(f"\nMejor configuración por MRR: {mejor}")

TABLA COMPARATIVA DE CONFIGURACIONES
configuracion  Recall_5  Precision_5    MRR  Latencia  N_preguntas
       A_BM25    0.7149       0.5632 0.6513    0.0072           38
  B_VECTORIAL    0.7785       0.7211 0.8026    0.7182           38
C_HIBRIDA_RRF    0.7917       0.6737 0.8092    0.5734           38

Mejor configuración por MRR: C_HIBRIDA_RRF


In [23]:
# ── Análisis por tipo de pregunta ──
print("\nRESULTADOS POR TIPO DE PREGUNTA:")
tipo_resumen = eval_df.groupby(["configuracion", "tipo_pregunta"]).agg(
    Recall_5 = ("recall_at_5", "mean"),
    MRR      = ("mrr",         "mean")
).round(3).reset_index()
print(tipo_resumen.to_string(index=False))


RESULTADOS POR TIPO DE PREGUNTA:
configuracion   tipo_pregunta  Recall_5   MRR
       A_BM25        aplicada     0.625 0.531
       A_BM25      conceptual     1.000 0.833
       A_BM25         factual     1.000 0.854
       A_BM25 multi-documento     0.310 0.476
       A_BM25   procedimental     0.400 0.400
  B_VECTORIAL        aplicada     0.875 0.875
  B_VECTORIAL      conceptual     1.000 1.000
  B_VECTORIAL         factual     1.000 0.938
  B_VECTORIAL multi-documento     0.369 0.571
  B_VECTORIAL   procedimental     0.400 0.400
C_HIBRIDA_RRF        aplicada     0.875 0.812
C_HIBRIDA_RRF      conceptual     1.000 1.000
C_HIBRIDA_RRF         factual     1.000 1.000
C_HIBRIDA_RRF multi-documento     0.440 0.607
C_HIBRIDA_RRF   procedimental     0.400 0.400


In [24]:
# ── Análisis por dificultad ──
print("\nRESULTADOS POR DIFICULTAD:")
dif_resumen = eval_df.groupby(["configuracion", "dificultad"]).agg(
    Recall_5 = ("recall_at_5", "mean"),
    MRR      = ("mrr",         "mean")
).round(3).reset_index()
print(dif_resumen.to_string(index=False))


RESULTADOS POR DIFICULTAD:
configuracion dificultad  Recall_5   MRR
       A_BM25       alta     0.700 0.756
       A_BM25       baja     1.000 1.000
       A_BM25      media     0.712 0.564
  B_VECTORIAL       alta     0.817 0.867
  B_VECTORIAL       baja     1.000 1.000
  B_VECTORIAL      media     0.742 0.750
C_HIBRIDA_RRF       alta     0.806 0.817
C_HIBRIDA_RRF       baja     1.000 1.000
C_HIBRIDA_RRF      media     0.773 0.795


## 11. Evaluación de alucinación (preguntas trampa)

In [25]:
print("EVALUACIÓN DE ALUCINACIÓN")
print("=" * 60)
print("Estas preguntas buscan documentos que NO existen en el corpus.")
print("El sistema NO debe inventar respuestas.\n")

for _, fila in preguntas_trampa.iterrows():
    pregunta = fila["pregunta"]
    print(f"PREGUNTA: {pregunta}")

    for config_nombre, search_fn in configuraciones.items():
        fragmentos = search_fn(pregunta, top_k=3)
        respuesta  = generar_respuesta(pregunta, fragmentos)
        print(f"\n  [{config_nombre}]")
        print(f"  Docs recuperados: {[r['id_documento'] for r in fragmentos]}")
        print(f"  Respuesta: {respuesta['respuesta'][:200]}")
    print("\n" + "=" * 60)

EVALUACIÓN DE ALUCINACIÓN
Estas preguntas buscan documentos que NO existen en el corpus.
El sistema NO debe inventar respuestas.

PREGUNTA: ¿Qué establece el DS 001-2020-MINAM sobre emisiones industriales?



  [A_BM25]
  Docs recuperados: ['DOC-015', 'DOC-008', 'DOC-021']
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication cred



  [B_VECTORIAL]
  Docs recuperados: ['DOC-008', 'DOC-012', 'DOC-010']
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication cred



  [C_HIBRIDA_RRF]
  Docs recuperados: ['DOC-015', 'DOC-008', 'DOC-016']
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication cred

PREGUNTA: ¿Qué sanciones económicas establece la Ley General del Ambiente para empresas contaminantes?



  [A_BM25]
  Docs recuperados: ['DOC-025', 'DOC-025', 'DOC-002']
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication cred



  [B_VECTORIAL]
  Docs recuperados: ['DOC-004', 'DOC-023', 'DOC-023']
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication cred



  [C_HIBRIDA_RRF]
  Docs recuperados: ['DOC-025', 'DOC-004', 'DOC-025']
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication cred



## 12. Evaluación de groundedness (respuestas sustentadas)

In [26]:
# Evaluar groundedness en una muestra de 10 preguntas con la mejor configuración
print(f"Evaluando groundedness con configuración: {mejor}")
print("=" * 60)

muestra = preguntas_eval.sample(min(10, len(preguntas_eval)), random_state=42)
groundedness_resultados = []

for _, fila in muestra.iterrows():
    pregunta = fila["pregunta"]

    if mejor == "A_BM25":
        fragmentos = search_bm25(pregunta)
    elif mejor == "B_VECTORIAL":
        fragmentos = search_vectorial(pregunta)
    else:
        fragmentos = search_hibrida(pregunta)

    resultado = generar_respuesta(pregunta, fragmentos)

    # Evaluación simple de groundedness:
    # ¿La respuesta menciona que no tiene información? → sustentada (no alucina)
    # ¿La respuesta parece inventada sin citar fuente? → no sustentada
    respuesta = resultado["respuesta"].lower()
    no_encontrado = any(phrase in respuesta for phrase in [
        "no se encuentra", "no está en", "no hay información",
        "no se menciona", "no dispongo", "no encontr"
    ])

    groundedness_resultados.append({
        "id_pregunta":   fila["id_pregunta"],
        "pregunta":      pregunta[:60],
        "latencia_seg":  resultado["latencia_seg"],
        "sin_informacion": no_encontrado,
        "respuesta_corta": resultado["respuesta"][:150]
    })

    print(f"Q: {pregunta[:60]}")
    print(f"  Latencia: {resultado['latencia_seg']}s")
    print(f"  Respuesta: {resultado['respuesta'][:150]}")
    print()

ground_df = pd.DataFrame(groundedness_resultados)
print(f"Latencia promedio: {ground_df['latencia_seg'].mean():.3f} seg")
print(f"Latencia P95:      {ground_df['latencia_seg'].quantile(0.95):.3f} seg")

Evaluando groundedness con configuración: C_HIBRIDA_RRF


Q: ¿Qué normas regulan la gestión de residuos sólidos en el sec
  Latencia: 1.973s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Cuál es el rol del MINAM en el Sistema Nacional de Gestión 
  Latencia: 1.945s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Cómo regula el DS 005-2026-MINAM la gestión de sustancias q
  Latencia: 1.748s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Cuál es el procedimiento para monitorear la calidad ambient
  Latencia: 2.201s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



...

Q: ¿Qué establece la Ley 32099 para la protección y uso sosteni
  Latencia: 1.893s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Cuáles son los Estándares de Calidad Ambiental (ECA) para a
  Latencia: 2.19s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Qué relación tienen los humedales con el cambio climático s
  Latencia: 2.099s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Cómo regula la NTS 073-2008-MINSA el manejo selectivo de re
  Latencia: 1.992s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod



Q: ¿Qué propone la RM 041-2014-MINAM sobre el Estándar de Calid
  Latencia: 2.247s
  Respuesta: Error al generar respuesta: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encod

Latencia promedio: 2.034 seg
Latencia P95:      2.226 seg


## 13. Guardado de resultados

In [27]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Resultados detallados ──
eval_path = REPORTS_DIR / f"evaluacion_configuraciones_{timestamp}.csv"
eval_df.to_csv(eval_path, index=False, encoding="utf-8-sig")
print(f"Resultados detallados: {eval_path}")

# ── Resumen comparativo ──
resumen_path = REPORTS_DIR / f"resumen_comparativo_{timestamp}.csv"
resumen.to_csv(resumen_path, index=False, encoding="utf-8-sig")
print(f"Resumen comparativo:   {resumen_path}")

# ── Reporte JSON ──
reporte = {
    "fecha":          datetime.now().isoformat(timespec="seconds"),
    "total_preguntas": len(preguntas_eval),
    "configuraciones": resumen.to_dict(orient="records"),
    "mejor_config_mrr": mejor,
    "top_k": TOP_K
}
json_path = REPORTS_DIR / f"reporte_final_{timestamp}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)
print(f"Reporte JSON:          {json_path}")

Resultados detallados: /content/drive/MyDrive/RAG_Normativa_Ambiental/experiments/resultados/evaluacion_configuraciones_20260609_070707.csv
Resumen comparativo:   /content/drive/MyDrive/RAG_Normativa_Ambiental/experiments/resultados/resumen_comparativo_20260609_070707.csv
Reporte JSON:          /content/drive/MyDrive/RAG_Normativa_Ambiental/experiments/resultados/reporte_final_20260609_070707.json


## 14. Resultado final

In [28]:
print("=" * 65)
print("RESUMEN FINAL DEL PROYECTO RAG")
print("=" * 65)
print(resumen.to_string(index=False))
print("=" * 65)
print(f"\nMejor configuración por MRR: {mejor}")
print("\nArchivos generados en experiments/resultados/")
print("\nSiguiente paso: construir demo en Streamlit (app/demo_streamlit.py)")

RESUMEN FINAL DEL PROYECTO RAG
configuracion  Recall_5  Precision_5    MRR  Latencia  N_preguntas
       A_BM25    0.7149       0.5632 0.6513    0.0072           38
  B_VECTORIAL    0.7785       0.7211 0.8026    0.7182           38
C_HIBRIDA_RRF    0.7917       0.6737 0.8092    0.5734           38

Mejor configuración por MRR: C_HIBRIDA_RRF

Archivos generados en experiments/resultados/

Siguiente paso: construir demo en Streamlit (app/demo_streamlit.py)


In [29]:
# DIAGNÓSTICO COMPLETO
print("=== 1. COLECCIONES EN QDRANT ===")
print([c.name for c in client.get_collections().collections])

print("\n=== 2. INFO DE LA COLECCIÓN ===")
info = client.get_collection(COLLECTION_NAME)
print(f"Total vectores: {info.points_count}")
print(f"Dimensión: {info.config.params.vectors.size}")

print("\n=== 3. MUESTRA DE PAYLOADS ===")
muestra = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False
)[0]
for p in muestra:
    print(f"\nID punto: {p.id}")
    print(f"Payload: {p.payload}")

print("\n=== 4. ID_DOCUMENTO EN PREGUNTAS ===")
print(preguntas_eval["doc_esperado"].head(5).tolist())

=== 1. COLECCIONES EN QDRANT ===
['normativa_ambiental_chunks_v1']

=== 2. INFO DE LA COLECCIÓN ===
Total vectores: 1057
Dimensión: 1024

=== 3. MUESTRA DE PAYLOADS ===

ID punto: 0
Payload: {'id_chunk': 'DOC-001_CHK-0001', 'id_documento': 'DOC-001', 'archivo_pdf': 'DOC-001_DS_004_2012_MINAM_IGAC.pdf', 'titulo_documento': 'Aprueban Disposiciones Complementarias para el Instrumento de Gestión Ambiental Correctivo - IGAC para la Formalización de Actividades de Pequeña Minería y Minería Artesanal en curso', 'tipo_norma': 'Decreto Supremo', 'numero_norma': 'Decreto Supremo N.º 004-2012-MINAM', 'entidad_emisora': 'MINAM', 'fecha_publicacion': '2012-09-06', 'tema_principal': 'Gestión ambiental', 'subtema': 'Minería y ambiente / IGAC', 'fuente_oficial': 'El Peruano', 'url_fuente': 'No determinado', 'estado_vigencia': 'No determinado', 'prioridad': 'Alta', 'tipo_chunk': 'preambulo', 'seccion': 'Preámbulo / considerandos', 'titulo_seccion': 'No determinado', 'capitulo': 'No determinado', 'anexo